# Capstone Case Study: Content Refresh Opportunity Scoring for FlyRank

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Saadali880/flyrank-ml-internship-saad/blob/main/work/notebooks/capstone.ipynb)

This notebook compiles the research, modeling, and recommendations for the Content Refresh Opportunity Scoring lane, acting as a case study for FlyRank's content-as-infrastructure platform. It demonstrates how a trained machine learning model can replace FlyRank's existing production system of hand-written product flags to optimize human editorial resources and halt search traffic decay.

## 1. Question

### The Research Question
How can we prioritize search content items for refresh, expansion, or optimization to maximize search traffic recovery under a limited editorial budget?

### Decision Support and Actions
- **Decision**: Which pages should a content review team examine first?
- **Action**: A content reviewer opens the prioritized queue and decides whether to refresh, expand, optimize search snippets, or monitor each page.
- **Cost of a Wrong Call**: Wasted editorial/reviewer time on healthy or low-value pages, or missing high-visibility pages that are actively decaying.

In [1]:
# Verification of the primary question and objectives
print("Primary Goal: Optimize editorial resources by prioritizing content items at risk of organic traffic decay.")
print("Target Lane: Lane 2 - Refresh / Content Opportunity Scoring")

Primary Goal: Optimize editorial resources by prioritizing content items at risk of organic traffic decay.
Target Lane: Lane 2 - Refresh / Content Opportunity Scoring


## 2. Data

### Telemetry and Features
- **Release**: FlyRank ML Internship dataset (v202603 / v202607).
- **Dataset size**: 30,000 rows × 44 columns, one row per content item across 32 clients.
- **Features included**: Google Search Console (GSC) metrics (impressions, clicks, average position, click-through rate over 90 days) and Google Analytics 4 (GA4) metrics (sessions, users, engagement rate, scroll rate).
- **Exclusions (Leakage Prevention)**: Column `trend_direction` and `trend_pct` were excluded because they directly define the target label. impressions, clicks, and sessions in the last 30 days and previous 30 days were also dropped since they overlap with the target outcome window.

In [2]:
import pandas as pd
import numpy as np

# Load raw starter dataset
df_raw = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')
print(f"Raw dataset shape: {df_raw.shape}")
print(f"Number of unique clients: {df_raw['client_id'].nunique()}")
print(f"Distribution of trend_direction:\n{df_raw['trend_direction'].value_counts()}")

Raw dataset shape: (30000, 44)
Number of unique clients: 32
Distribution of trend_direction:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64


## 3. Methodology

### Train/Test Validation Split
- **Split Strategy**: Grouped train/test split on `client_id` (80% train, 20% test, `random_state=42`).
- **Why it is honest**: Prevents data leakage of client-specific characteristics. All pages belonging to a client are either entirely in the train split or entirely in the test split, evaluating the model's ability to generalize to unseen clients in production.
- **Leakage Checks**: Verified that no features represent future outcome-window measurements.

In [3]:
from sklearn.model_selection import GroupShuffleSplit

df_feat = pd.read_csv('../../data/processed/refresh_feature_vector.csv')
y = df_feat['is_declining_label'].to_numpy()
groups = df_feat['client_id'].to_numpy()

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(df_feat, y, groups))

print(f"Train split: {len(train_idx)} rows, {df_feat.iloc[train_idx]['client_id'].nunique()} clients, decline base rate: {df_feat.iloc[train_idx]['is_declining_label'].mean():.3%}")
print(f"Test split:  {len(test_idx)} rows, {df_feat.iloc[test_idx]['client_id'].nunique()} clients, decline base rate: {df_feat.iloc[test_idx]['is_declining_label'].mean():.3%}")

Train split: 23837 rows, 25 clients, decline base rate: 55.011%
Test split:  6163 rows, 7 clients, decline base rate: 51.095%


## 4. Results (vs baseline)

### Performance on Holdout Client Split
- **Champion Model**: Logistic Regression (with L2 regularization, balanced class weights) achieves a **Precision@50 of 0.740** (vs. baseline score **0.300**). This represents a **2.47x lift** over the baseline.
- **Linear Weight Robustness**: Tree models (Random Forest, Decision Tree) suffer from overfitting to training client scales (Precision@50 of 0.34-0.66), failing to generalize as well as Logistic Regression.

In [4]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

# Preprocess and scale features
df_proc = df_feat.copy()
impute_cols = ['word_count', 'search_volume', 'avg_position', 'cpc', 'competition']
for col in impute_cols:
    train_subset = df_proc.iloc[train_idx]
    non_zero = train_subset.loc[train_subset[col] > 0, col]
    median_val = non_zero.median() if len(non_zero) > 0 else 0
    df_proc.loc[df_proc[col] <= 0, col] = median_val

leakage_cols = [
    'trend_direction', 'trend_pct', 'is_declining_label', 'content_id', 'client_id',
    'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d',
    'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d'
]
features_df = df_proc.drop(columns=[c for c in leakage_cols if c in df_proc.columns])
cat_cols = features_df.select_dtypes(include=['object', 'category']).columns.tolist()
features_df = pd.get_dummies(features_df, columns=cat_cols, drop_first=True)
for col in features_df.select_dtypes(include=['bool']).columns:
    features_df[col] = features_df[col].astype(float)

X = features_df.to_numpy()
feature_names = features_df.columns.tolist()
X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

def precision_at_k(y_true, y_prob, k):
    top_indices = np.argsort(y_prob)[::-1][:k]
    return np.mean(y_true[top_indices])

# Champion Logistic Regression
lr = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
lr.fit(X_train_scaled, y_train)
y_prob_lr = lr.predict_proba(X_test_scaled)[:, 1]

# Baseline Score
test_df = df_proc.iloc[test_idx].copy()
stale_flag = (test_df['days_since_last_update'] >= 90).astype(int)
visible_flag = (test_df['impressions_90d'] >= 500).astype(int)
striking_flag = ((test_df['avg_position'] > 3) & (test_df['avg_position'] <= 15)).astype(int)
rule_match = stale_flag * visible_flag * striking_flag
freshness_rank = test_df['days_since_last_update'].rank(pct=True)
visibility_rank = np.log1p(test_df['impressions_90d']).rank(pct=True)
pos_opp = 1 - (test_df['avg_position'].clip(3, 15) - 3) / 12
baseline_score = rule_match * (0.4 * freshness_rank + 0.4 * visibility_rank + 0.2 * pos_opp)

print("=== Performance Comparison on Unseen Clients ===")
print(f"Logistic Regression  - ROC AUC: {roc_auc_score(y_test, y_prob_lr):.4f} | Precision@50: {precision_at_k(y_test, y_prob_lr, 50):.4%}")
print(f"Baseline Heuristic   - ROC AUC: {roc_auc_score(y_test, baseline_score):.4f} | Precision@50: {precision_at_k(y_test, baseline_score, 50):.4%}")

C:\Users\hassa\AppData\Local\Temp\ipykernel_22044\3543669744.py:22: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = features_df.select_dtypes(include=['object', 'category']).columns.tolist()


=== Performance Comparison on Unseen Clients ===
Logistic Regression  - ROC AUC: 0.6162 | Precision@50: 76.0000%
Baseline Heuristic   - ROC AUC: 0.4990 | Precision@50: 30.0000%


## 5. Limitations

### Key Boundaries and Limits
1. **Stability Paradox**: High-visibility pages have a lower baseline rate of decline but are critical to monitor due to the massive scale of traffic loss if they slip.
2. **Observational vs. Causal**: The model finds correlation with decline, not causal proof that refreshing will recover rankings. Matched A/B tests are required.
3. **Domain Blindness**: Purely telemetry-based; the model is blind to page relevance or deprecation.
4. **Seasonality and Core Updates**: External market trends can confound decline risk.
5. **Confounded Engagement**: Low GA4 engagement can result from site code errors, not text quality.

In [5]:
# Print out top coefficients driving risk to illustrate the Stability Paradox
coefs = lr.coef_[0]
coef_df = pd.DataFrame({'Feature': feature_names, 'Coef': coefs})
print("Top Positive Coefficients (Risk Drivers):")
print(coef_df.sort_values('Coef', ascending=False).head(3).to_string(index=False))
print("\nTop Negative Coefficients (Risk Reducers):")
print(coef_df.sort_values('Coef', ascending=True).head(3).to_string(index=False))

Top Positive Coefficients (Risk Drivers):
            Feature     Coef
log_impressions_90d 1.542372
       sessions_90d 0.698422
impression_tier_low 0.575435

Top Negative Coefficients (Risk Reducers):
       Feature      Coef
     users_90d -0.808038
log_clicks_90d -0.603340
  avg_position -0.552132


## 6. Ranked recommendations

### Action Playbook
- **Scoring Formula**: Blended Score = `70% * model_probability + 30% * normalized_baseline_score` (scaled 0-100).
- **Suggested Actions**:
  1. `expand_and_refresh` (Thin content with high demand: word_count < 1200, impressions >= 250)
  2. `refresh_and_review_ctr` (Striking distance, CTR < 0.5%, impressions >= 500, average position 1-20)
  3. `refresh_and_review_engagement` (Sessions >= 30, scroll or engagement rate < 30%)
  4. `refresh` (Other decaying visible content)
  5. `monitor` (Healthy or low-demand pages)

In [6]:
# Load generated action queue and show action mix
q_df = pd.read_csv('../outputs/refresh_queue.csv')
print("Suggested Action Mix:")
print(q_df['suggested_action'].value_counts())
print("\nConfidence Mix:")
print(q_df['confidence'].value_counts())

Suggested Action Mix:


suggested_action
monitor                          13069
refresh                           8207
refresh_and_review_ctr            6655
refresh_and_review_engagement     1987
expand_and_refresh                  82
Name: count, dtype: int64

Confidence Mix:
confidence
low       15000
medium    11424
high       3576
Name: count, dtype: int64


## 7. Artifacts the paper embeds

We verified that key charts representing feature importances, action distributions, and model performance have been exported and saved under `work/figures/` for display in the static page.

In [7]:
import os
print("Figures available:")
print(os.listdir('../figures'))

Figures available:
['action_mix.png', 'confidence_mix.png', 'score_distribution.png', 'top_feature_importance.png', 'top_reason_codes.png']


## 8. Showcase Demo Outline (5-Minute Presentation)

This outline structures a 5-minute presentation for the Week-8 showcase, summarizing the research and outcomes of this project.

### Slide 1: The Question & Problem (1 Minute)
- **The Hook**: Organic search traffic decays silently. For large portfolios, manual text audits are too expensive to run on every page.
- **The Context**: FlyRank builds content-as-infrastructure, publishing and optimizing content at scale. The production platform uses hand-written rules (heuristic health/priority scores) to flag pages for update, but these rules cannot scale to complex, multi-dimensional signals.
- **The Question**: How can we build an empirical opportunity scoring model to rank content pages for revision, maximizing traffic protection under a limited copywriter budget?

### Slide 2: The Methodology & Validation (1 Minute)
- **Telemetry**: Built on 30,000 unique page-level rows across 32 clients. Features merge Google Search Console (GSC) query visibility with Google Analytics 4 (GA4) post-click engagement telemetry.
- **The Split (Leakage Prevention)**: Implemented an honest Grouped Train/Test split on `client_id` (80% train, 20% test). Pages from the same client are kept together, ensuring the model is evaluated on entirely unseen client profiles.
- **Model Space**: Compared Logistic Regression (L2 regularization), Decision Trees, and Random Forests.

### Slide 3: One Key Chart - Feature Importance (1 Minute)
- **The Visual**: Coefficient weight chart from our champion regularized linear model (`top_feature_importance.png`).
- **The Insight (Stability Paradox)**: Historical GSC impressions have a large positive coefficient (driving decline risk). This represents the *Stability Paradox*: high-performing assets have the most absolute room to fall. Baseline heuristics fail to prioritize them because they only look at poor health scores, while our model identifies them as critical traffic protection opportunities.

### Slide 4: One Honest Result (1 Minute)
- **Performance**: Regularized Logistic Regression is the champion, achieving **74.0% Precision@50** on the unseen test split.
- **The Lift**: This is a **2.47x efficiency lift** over FlyRank's existing heuristic rule (30.0% Precision@50). Out of 50 editor reviews, 37 represent true decay vs. only 15 under the heuristic.
- **Honest Framing**: Complex tree ensembles overfit to client traffic scales (ROC AUC drops on holdout). The linear model generalized far better to new client domains.

### Slide 5: One Recommendation (1 Minute)
- **The Action Playbook**: Blend the model's decay probability with baseline opportunity scores (`70% Model + 30% Heuristic`) to priority-rank pages.
- **Action Routing**: Automatically categorize high-confidence queue items into actionable copyediting playbooks: `refresh` for standard text edits, `refresh_and_review_ctr` for search snippet fixes, and `monitor` for healthy evergreen assets.
- **Retrain Triggers**: Monitor the queue; if holdout Precision@50 drops below 50% or editorial rejection rate exceeds 35%, trigger model recalibration.

## 9. Shareable Cuts

### Cut 1: Short Social Post (Methodology-focused)
"How do you scale search traffic protection across thousands of pages without wasting editorial budgets?

At FlyRank, we replaced hand-written product heuristics with regularized machine learning classifiers to prioritize content refreshes. The key? An honest client-grouped validation split to prevent leakage, showing that deep tree models overfit to domain scale, while regularized Logistic Regression generalized beautifully to unseen sites—delivering a **2.47x lift** in Precision@50 (from 30% to 74%).

Crucial discovery: The Stability Paradox. The pages with the highest impressions carry the greatest statistical risk of decay. Protecting top-performers yields a far higher ROI than editing bottom-ranked content.

Check out the full case study: [deployed URL]"

### Cut 2: Employer-Facing Summary (3-Sentence)
I built an empirical content refresh opportunity scoring engine to prioritize organic search traffic protection on FlyRank's content-as-infrastructure platform. Using 30,000 rows of Google Search Console and GA4 telemetry across 32 clients, I validated the framework using a client-grouped split to ensure generalization to unseen domains. The champion regularized linear model achieved a 74% Precision@50, representing a 2.47x lift in editorial resource efficiency over the existing production baseline rule.

## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.